In [ ]:
# ==========================================
# MBTI Personality Classification using Logistic Regression (Old scikit-learn compatible)
# ==========================================

import pandas as pd
import numpy as np
import re
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ===============================
# LOAD DATA
# ===============================
print("Loading dataset...")
df = pd.read_csv("MBTI 500.csv")
df.columns = df.columns.str.strip()

# ===============================
# LIGHT CLEANING
# ===============================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'\s+', ' ', text)     # normalize spaces
    return text.strip()

df['posts'] = df['posts'].apply(clean_text)
df = df[df['posts'].str.len() > 20]

# ===============================
# LABEL ENCODING
# ===============================
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['type'])
num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)

# Save label encoder
joblib.dump(label_encoder, "mbti_label_encoder.pkl")

# ===============================
# TRAIN TEST SPLIT
# ===============================
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['posts'].tolist(),
    df['labels'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['labels']
)

# ===============================
# TF-IDF VECTORIZATION
# ===============================
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

# ===============================
# LOGISTIC REGRESSION MODEL (OLD VERSION COMPATIBLE)
# ===============================
model = LogisticRegression(max_iter=500)  # works with older scikit-learn
model.fit(X_train, train_labels)

# ===============================
# EVALUATION
# ===============================
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(test_labels, y_pred), 4))
print("\nClassification Report:\n")
print(classification_report(
    test_labels,
    y_pred,
    target_names=label_encoder.classes_
))

# ===============================
# SAVE MODEL AND VECTORIZER
# ===============================
joblib.dump(model, "mbti_logistic_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("\nModel and vectorizer saved successfully!")